In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
import xgboost as xgb

In [5]:
df = pd.read_csv("joined_data.csv")
df


,observation_date,gdp_growth_rate,unemployment_rate,cpi_commodities,covid_dummy
0,1956-01-01,-1.5,4.033333,31.166667,0
1,1956-04-01,3.3,4.200000,31.400000,0
2,1956-07-01,-0.3,4.133333,31.766667,0
3,1956-10-01,6.8,4.133333,32.033333,0
4,1957-01-01,2.6,3.933333,32.233333,0
...,...,...,...,...,...
275,2024-10-01,1.9,4.133333,222.827333,0
276,2025-01-01,-0.6,4.133333,224.413000,0
277,2025-04-01,3.8,4.200000,223.847000,0
278,2025-07-01,4.4,4.333333,225.254000,0


In [6]:
# getting rid of covid quarters and covid dummy variable
df = df.drop(columns=['covid_dummy'])
df = df[~df['observation_date'].isin(['2020-04-01', '2020-07-01'])]
df.head()

,observation_date,gdp_growth_rate,unemployment_rate,cpi_commodities
0,1956-01-01,-1.5,4.033333,31.166667
1,1956-04-01,3.3,4.200000,31.400000
2,1956-07-01,-0.3,4.133333,31.766667
3,1956-10-01,6.8,4.133333,32.033333
4,1957-01-01,2.6,3.933333,32.233333


In [7]:
# dropping dates before 1990
df = df[df['observation_date'] >= '1990-01-01'].copy().reset_index(drop=True)
df

,observation_date,gdp_growth_rate,unemployment_rate,cpi_commodities
0,1990-01-01,4.4,5.300000,120.600000
1,1990-04-01,1.5,5.333333,121.200000
2,1990-07-01,0.3,5.700000,123.233333
3,1990-10-01,-3.6,6.133333,126.033333
4,1991-01-01,-1.9,6.600000,125.900000
...,...,...,...,...
137,2024-10-01,1.9,4.133333,222.827333
138,2025-01-01,-0.6,4.133333,224.413000
139,2025-04-01,3.8,4.200000,223.847000
140,2025-07-01,4.4,4.333333,225.254000


In [8]:
# Differencing CPI data to make it stationary
df['cpi_diff'] = df['cpi_commodities'].diff()

#making lagged features 
for lag in [1, 2, 4]:
    df[f'unemployment_lag{lag}'] = df['unemployment_rate'].shift(lag)
    df[f'gdp_growth_lag{lag}'] = df['gdp_growth_rate'].shift(lag)
    df[f'cpi_diff_lag{lag}'] = df['cpi_diff'].shift(lag)

df = df.dropna().reset_index(drop=True)

df


,observation_date,gdp_growth_rate,unemployment_rate,cpi_commodities,cpi_diff,unemployment_lag1,gdp_growth_lag1,cpi_diff_lag1,unemployment_lag2,gdp_growth_lag2,cpi_diff_lag2,unemployment_lag4,gdp_growth_lag4,cpi_diff_lag4
0,1991-04-01,3.2,6.833333,126.366667,0.466667,6.600000,-1.9,-0.133333,6.133333,-3.6,2.800000,5.333333,1.5,0.600000
1,1991-07-01,2.0,6.866667,126.800000,0.433333,6.833333,3.2,0.466667,6.600000,-1.9,-0.133333,5.700000,0.3,2.033333
2,1991-10-01,1.4,7.100000,127.500000,0.700000,6.866667,2.0,0.433333,6.833333,3.2,0.466667,6.133333,-3.6,2.800000
3,1992-01-01,4.9,7.366667,127.866667,0.366667,7.100000,1.4,0.700000,6.866667,2.0,0.433333,6.600000,-1.9,-0.133333
4,1992-04-01,4.4,7.600000,128.666667,0.800000,7.366667,4.9,0.366667,7.100000,1.4,0.700000,6.833333,3.2,0.466667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,2024-10-01,1.9,4.133333,222.827333,0.611000,4.166667,3.3,-1.212000,3.966667,3.6,0.001667,3.800000,3.4,-0.545333
133,2025-01-01,-0.6,4.133333,224.413000,1.585667,4.133333,1.9,0.611000,4.166667,3.3,-1.212000,3.833333,0.8,-0.269667
134,2025-04-01,3.8,4.200000,223.847000,-0.566000,4.133333,-0.6,1.585667,4.133333,1.9,0.611000,3.966667,3.6,0.001667
135,2025-07-01,4.4,4.333333,225.254000,1.407000,4.200000,3.8,-0.566000,4.133333,-0.6,1.585667,4.166667,3.3,-1.212000


In [9]:
# chronologically split train and test data 
train = df[df['observation_date'] <= '2019-10-01'].copy()
test = df[df['observation_date'] >= '2021-01-01'].copy()
feature_cols = ['gdp_growth_rate','cpi_diff','unemployment_lag1', 'unemployment_lag2', 'unemployment_lag4','gdp_growth_lag1',  'gdp_growth_lag2',  'gdp_growth_lag4', 'cpi_diff_lag1',    'cpi_diff_lag2',    'cpi_diff_lag4']

# splitting features

X_train = train[feature_cols]
y_train = train['unemployment_rate']
X_test  = test[feature_cols]
y_test  = test['unemployment_rate']

# scaling our train data
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [10]:
results = {}

# baseline model, just mean
baseline_pred = np.full(len(y_test), y_train.mean())
results['Baseline (mean)'] = {
    'RMSE': np.sqrt(mean_squared_error(y_test, baseline_pred)),
    'R2': r2_score(y_test, baseline_pred)
}

# Lagged linear regression
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)
results['Linear Regression'] = {
    'RMSE': np.sqrt(mean_squared_error(y_test, lr_pred)),
    'R2': r2_score(y_test, lr_pred)
}

rf = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train_sc, y_train)
rf_pred = rf.predict(X_test_sc)
results['Random Forest'] = {
    'RMSE': np.sqrt(mean_squared_error(y_test, rf_pred)),
    'R2': r2_score(y_test, rf_pred)
}

print(pd.DataFrame(results).T.round(4))

                     RMSE      R2
Baseline (mean)    1.8129 -5.0783
Linear Regression  0.3713  0.7450
Random Forest      0.3816  0.7307


In [13]:
# trying xgboost model 

xgb_model = xgb.XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42, verbosity=0)
xgb_model.fit(X_train_sc, y_train)
xgb_pred = xgb_model.predict(X_test_sc)
results['XGBoost'] = {
    'RMSE': np.sqrt(mean_squared_error(y_test, xgb_pred)),
    'R2': r2_score(y_test, xgb_pred)
}

print(pd.DataFrame(results).T.round(4))

                     RMSE      R2
Baseline (mean)    1.8129 -5.0783
Linear Regression  0.3713  0.7450
Random Forest      0.3816  0.7307
XGBoost            0.4073  0.6932


In [16]:
# trying ridge regression 
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_sc, y_train)
ridge_pred = ridge.predict(X_test_sc)
results['Ridge'] = {
    'RMSE': np.sqrt(mean_squared_error(y_test, ridge_pred)),
    'R2':   r2_score(y_test, ridge_pred)
}

print(pd.DataFrame(results).T.round(4))

                     RMSE      R2
Baseline (mean)    1.8129 -5.0783
Linear Regression  0.3713  0.7450
Random Forest      0.3816  0.7307
XGBoost            0.4073  0.6932
Ridge              0.3307  0.7977


# Ridge regression performed best with lagged features.